# Day 9: Node.js and Express
**The Node runtime, npm and supply chain hygiene, Express CRUD APIs, and testing with Postman**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.
* Sections marked **NEW (v4)** were added in program v4.

## 1. Setup
Google Colab already has Node.js and npm. `%%writefile` saves a file and `!node` runs it.

**VS Code:** install Node.js LTS from nodejs.org, then run the same files with `node file.js` in the terminal.

In [ ]:
# ===== Option A: Google Colab (active) =====
!node --version
!npm --version

# ===== Option B: VS Code =====
# In the VS Code terminal:
#   node --version
#   npm --version

## 2. The Node runtime: modules and files (Session 1)

In [ ]:
%%writefile store.js
const fs = require("fs/promises");

let tasks = [];
let nextId = 1;

function add(title) {
  const task = { id: nextId++, title, done: false };
  tasks.push(task);
  return task;
}
const list = () => [...tasks];
async function save(path) {
  await fs.writeFile(path, JSON.stringify(tasks, null, 2));
  return path;
}
module.exports = { add, list, save };

In [ ]:
%%writefile main.js
const { add, list, save } = require("./store");
const fs = require("fs/promises");

async function main() {
  ["Read post-read", "Install Postman", "Run npm audit"].forEach(add);
  console.log("Added", list().length, "tasks");
  const path = await save("tasks.json");
  console.log("Saved to", path);
  console.log(await fs.readFile(path, "utf8"));
  console.log("Node", process.version, "on", process.platform);
}
main();

In [ ]:
# ===== Option A: Google Colab (active) =====
!node main.js

# ===== Option B: VS Code =====
# node main.js

## 3. npm and package.json

In [ ]:
%%writefile package.json
{
  "name": "tasks-api",
  "version": "1.0.0",
  "private": true,
  "scripts": {
    "start": "node src/server.js",
    "dev": "nodemon src/server.js",
    "test": "node tests/api_test.js",
    "audit": "npm audit --audit-level=high"
  },
  "dependencies": {
    "express": "^5.1.0"
  },
  "devDependencies": {
    "nodemon": "^3.1.0"
  }
}

In [ ]:
# ===== Option A: Google Colab (active) =====
!npm install --silent
!npm ls --depth=0

# ===== Option B: VS Code =====
# npm install
# npm ls --depth=0

## 4. NEW (v4): npm supply chain hygiene
`package.json` holds **ranges**. `package-lock.json` holds the **exact** versions that were installed. Commit the lockfile, never `node_modules`. This script explains each range and shows the locked version when a lockfile exists.

In [ ]:
%%writefile check_lock.js
const fs = require("fs");
const pkg = JSON.parse(fs.readFileSync("package.json", "utf8"));
const lock = fs.existsSync("package-lock.json")
  ? JSON.parse(fs.readFileSync("package-lock.json", "utf8")) : null;

function explain(range) {
  if (range.startsWith("^")) return `same major (${range.slice(1).split(".")[0]}.x.x)`;
  if (range.startsWith("~")) return `same minor (${range.slice(1).split(".").slice(0, 2).join(".")}.x)`;
  return "exact version only";
}

const all = { ...pkg.dependencies, ...pkg.devDependencies };
for (const [name, range] of Object.entries(all)) {
  const locked = lock?.packages?.[`node_modules/${name}`]?.version ?? "not installed";
  const dev = name in (pkg.devDependencies || {}) ? " (dev)" : "";
  console.log(`${name.padEnd(9)} range ${range.padEnd(8)} -> ${explain(range).padEnd(22)} locked ${locked}${dev}`);
}
console.log(lock ? "Lockfile found: commit it." : "No lockfile yet: run npm install, then commit package-lock.json.");

In [ ]:
# ===== Option A: Google Colab (active) =====
!node check_lock.js

# ===== Option B: VS Code =====
# node check_lock.js

### npm audit
The real command needs internet access to the npm registry. Run it in Colab or VS Code:

In [ ]:
# ===== Option A: Google Colab (active) =====
!npm audit
!npm audit --audit-level=high; echo "exit code: $?"

# ===== Option B: VS Code =====
# npm audit
# npm audit --audit-level=high

To practise reading audit output offline, the next cell writes a **made-up sample** in the same JSON shape as `npm audit --json`. The package names are fictional. Then a script turns it into vulnerability report rows (the template from Day 7).

In [ ]:
%%writefile sample_audit.json
{
  "vulnerabilities": {
    "example-parser": {"name": "example-parser", "severity": "high", "isDirect": false,
                       "via": [{"title": "Sample: prototype pollution in example-parser"}],
                       "fixAvailable": {"name": "example-parser", "version": "2.4.1"}},
    "demo-logger": {"name": "demo-logger", "severity": "moderate", "isDirect": true,
                    "via": [{"title": "Sample: log injection in demo-logger"}],
                    "fixAvailable": true},
    "toy-template": {"name": "toy-template", "severity": "moderate", "isDirect": false,
                     "via": [{"title": "Sample: regular expression denial of service"}],
                     "fixAvailable": false}
  }
}

In [ ]:
%%writefile audit_report.js
const fs = require("fs");
const file = process.argv[2] || "sample_audit.json";
const { vulnerabilities } = JSON.parse(fs.readFileSync(file, "utf8"));
const levels = ["critical", "high", "moderate", "low"];
const counts = Object.fromEntries(levels.map(l => [l, 0]));

const rows = Object.values(vulnerabilities).map((v, i) => {
  counts[v.severity] += 1;
  const fix = v.fixAvailable === false ? "no fix yet: consider overrides or replacing"
    : v.fixAvailable === true ? "npm audit fix" : `upgrade to ${v.fixAvailable.version}`;
  const title = typeof v.via[0] === "string" ? v.via[0] : v.via[0].title;
  return `V-${String(i + 1).padStart(2, "0")} | npm audit | ${v.severity} | ${v.name}${v.isDirect ? "" : " (indirect)"} | ${title} | ${fix}`;
});

console.log(levels.map(l => `${l} ${counts[l]}`).join(" | "));
console.log("ID | tool | severity | affected package | finding | proposed fix");
rows.forEach(r => console.log(r));
process.exitCode = counts.critical + counts.high > 0 ? 1 : 0;
console.log(process.exitCode ? "Gate: FAIL (high or critical found)" : "Gate: PASS");

In [ ]:
# ===== Option A: Google Colab (active) =====
!node audit_report.js sample_audit.json
# With real data:  !npm audit --json > audit.json; node audit_report.js audit.json

# ===== Option B: VS Code =====
# node audit_report.js sample_audit.json

## 5. Express: routes, controllers and CRUD (Session 2)
The app is split into `src/app.js` (middleware and routers), `src/routes`, `src/controllers` and an async `src/store.js` that simulates a database.

In [ ]:
import os
os.makedirs("src/routes", exist_ok=True)
os.makedirs("src/controllers", exist_ok=True)
os.makedirs("tests", exist_ok=True)
print("Folders ready:", sorted(os.listdir("src")))

In [ ]:
%%writefile src/store.js
// Async in-memory store: every function returns a promise, like a real database driver.
const wait = (ms) => new Promise(r => setTimeout(r, ms));
let tasks = [];
let nextId = 1;

module.exports = {
  async list(filter = {}) {
    await wait(5);
    if (filter.simulate === "down") throw new Error("database unavailable");
    return filter.done === undefined ? tasks : tasks.filter(t => String(t.done) === filter.done);
  },
  async get(id) { await wait(5); return tasks.find(t => t.id === Number(id)); },
  async create(title) { await wait(5); const t = { id: nextId++, title, done: false }; tasks.push(t); return t; },
  async update(id, data) {
    const t = await this.get(id);
    if (t) Object.assign(t, { title: data.title ?? t.title, done: data.done ?? t.done });
    return t;
  },
  async remove(id) {
    const t = await this.get(id);
    if (t) tasks = tasks.filter(x => x !== t);
    return Boolean(t);
  },
};

In [ ]:
%%writefile src/controllers/tasks.js
const store = require("../store");

exports.list = async (req, res) => res.json(await store.list(req.query));

exports.getOne = async (req, res) => {
  const task = await store.get(req.params.id);
  if (!task) return res.status(404).json({ error: "Task not found" });
  res.json(task);
};

exports.create = async (req, res) => {
  const { title } = req.body ?? {};
  if (!title || typeof title !== "string" || title.length > 100) {
    return res.status(400).json({ error: "title is required (1 to 100 characters)" });
  }
  res.status(201).json(await store.create(title));
};

exports.update = async (req, res) => {
  const task = await store.update(req.params.id, req.body ?? {});
  if (!task) return res.status(404).json({ error: "Task not found" });
  res.json(task);
};

exports.remove = async (req, res) => {
  if (!(await store.remove(req.params.id))) return res.status(404).json({ error: "Task not found" });
  res.status(204).end();
};

In [ ]:
%%writefile src/routes/tasks.js
const router = require("express").Router();
const c = require("../controllers/tasks");

router.get("/", c.list);
router.get("/:id", c.getOne);
router.post("/", c.create);
router.put("/:id", c.update);
router.delete("/:id", c.remove);

module.exports = router;

In [ ]:
%%writefile src/app.js
const express = require("express");
const app = express();
app.use(express.json());

// Logging middleware: method, path, status, time
app.use((req, res, next) => {
  const start = Date.now();
  res.on("finish", () => console.log(`  ${req.method} ${req.originalUrl} ${res.statusCode} ${Date.now() - start}ms`));
  next();
});

app.get("/health", (req, res) => res.json({ status: "ok" }));
app.use("/tasks", require("./routes/tasks"));

// Catch-all 404 for unknown routes
app.use((req, res) => res.status(404).json({ error: "Route not found" }));

// Error middleware (four arguments). Express 5 sends rejected promises from async routes here.
app.use((err, req, res, next) => {
  console.error("  error:", err.message);
  res.status(500).json({ error: "Something went wrong" });
});

module.exports = app;

In [ ]:
%%writefile src/server.js
const app = require("./app");
const PORT = process.env.PORT || 3000;
app.listen(PORT, () => console.log(`Tasks API on http://localhost:${PORT}`));

## 6. Test the API from a script (Session 3)
The test starts the app on a free port, calls every endpoint with `fetch`, checks the status codes, and stops the server. It needs Express installed (section 3).

In [ ]:
%%writefile tests/api_test.js
const app = require("../src/app");

async function run() {
  const server = app.listen(0);
  const base = `http://127.0.0.1:${server.address().port}`;
  const call = async (method, path, body) => {
    const res = await fetch(base + path, { method, headers: { "Content-Type": "application/json" },
                                           body: body ? JSON.stringify(body) : undefined });
    return { status: res.status, body: res.status === 204 ? null : await res.json() };
  };
  const checks = [
    ["POST /tasks", await call("POST", "/tasks", { title: "Learn Express" }), 201],
    ["POST /tasks without title", await call("POST", "/tasks", {}), 400],
    ["GET /tasks", await call("GET", "/tasks"), 200],
    ["PUT /tasks/1", await call("PUT", "/tasks/1", { done: true }), 200],
    ["GET /tasks?done=true", await call("GET", "/tasks?done=true"), 200],
    ["GET /tasks/999", await call("GET", "/tasks/999"), 404],
    ["GET /tasks?simulate=down", await call("GET", "/tasks?simulate=down"), 500],
    ["DELETE /tasks/1", await call("DELETE", "/tasks/1"), 204],
    ["GET /nope", await call("GET", "/nope"), 404],
  ];
  let passed = 0;
  for (const [name, r, expected] of checks) {
    const ok = r.status === expected;
    passed += ok;
    console.log(`${ok ? "PASS" : "FAIL"} ${name} -> ${r.status}${r.body ? " " + JSON.stringify(r.body) : ""}`);
  }
  console.log(`${passed}/${checks.length} checks passed`);
  server.close();
}
run();

In [ ]:
# ===== Option A: Google Colab (active) =====
!node tests/api_test.js

# ===== Option B: VS Code =====
# npm test
# To run the server for Postman:  npm run dev   (then use http://localhost:3000)

### TODO 1: Add `GET /tasks/stats` returning total, done and pending counts. Register it **before** `/:id` in the router so it matches first, then add a check for it in `tests/api_test.js`.

In [ ]:
# TODO: write your code here

## 7. A ready-made Postman collection
This cell writes `tasks_api.postman_collection.json`. In Postman: **Import**, choose the file, then set the collection variable `baseUrl` to `http://localhost:3000`.

In [ ]:
import json

def req(name, method, path, body=None, status=200):
    item = {"name": name, "request": {"method": method,
            "header": [{"key": "Content-Type", "value": "application/json"}],
            "url": {"raw": "{{baseUrl}}" + path, "host": ["{{baseUrl}}"],
                    "path": [p for p in path.split("?")[0].split("/") if p]}},
            "event": [{"listen": "test", "script": {"type": "text/javascript",
                       "exec": [f'pm.test("status is {status}", () => pm.response.to.have.status({status}));']}}]}
    if body is not None:
        item["request"]["body"] = {"mode": "raw", "raw": json.dumps(body, indent=2)}
    return item

collection = {
    "info": {"name": "Day 9 Tasks API", "schema": "https://schema.getpostman.com/json/collection/v2.1.0/collection.json"},
    "variable": [{"key": "baseUrl", "value": "http://localhost:3000"}],
    "item": [
        req("Health", "GET", "/health"),
        req("Create task", "POST", "/tasks", {"title": "Learn Express"}, 201),
        req("Create task without title", "POST", "/tasks", {}, 400),
        req("List tasks", "GET", "/tasks"),
        req("Get task 1", "GET", "/tasks/1"),
        req("Update task 1", "PUT", "/tasks/1", {"done": True}),
        req("Delete task 1", "DELETE", "/tasks/1", status=204),
        req("Get missing task", "GET", "/tasks/999", status=404),
    ],
}
with open("tasks_api.postman_collection.json", "w") as f:
    json.dump(collection, f, indent=2)
print("Saved with", len(collection["item"]), "requests")

# ===== Option A: Google Colab, download the file (uncomment to use) =====
# from google.colab import files
# files.download("tasks_api.postman_collection.json")

# ===== Option B: VS Code =====
# The file is already in your project folder.

## 8. SmartDesk step 8: notification service
A small Express service that records ticket notifications. Sending is simulated: it logs the message instead of calling an SMS or email provider. Any provider key would come from `process.env`, never from the code.

In [ ]:
%%writefile notify_app.js
const express = require("express");
const app = express();
app.use(express.json());

const CHANNELS = ["email", "sms", "push"];
const sent = [];

app.get("/health", (req, res) => res.json({ status: "ok", service: "smartdesk-notify" }));

app.post("/notifications", async (req, res) => {
  const { ticketId, channel, message } = req.body ?? {};
  const errors = [];
  if (!Number.isInteger(ticketId)) errors.push("ticketId must be an integer");
  if (!CHANNELS.includes(channel)) errors.push(`channel must be one of ${CHANNELS.join(", ")}`);
  if (!message || message.length > 300) errors.push("message is required (max 300 characters)");
  if (errors.length) return res.status(400).json({ errors });
  const item = { id: sent.length + 1, ticketId, channel, message, sentAt: new Date().toISOString() };
  sent.push(item);                               // simulated send
  res.status(201).json(item);
});

app.get("/notifications", (req, res) => {
  const { ticketId } = req.query;
  res.json(ticketId ? sent.filter(n => n.ticketId === Number(ticketId)) : sent);
});

app.use((req, res) => res.status(404).json({ error: "Route not found" }));
app.use((err, req, res, next) => res.status(500).json({ error: "Something went wrong" }));
module.exports = app;

if (require.main === module) {
  const PORT = process.env.PORT || 4000;
  app.listen(PORT, () => console.log(`SmartDesk notify service on http://localhost:${PORT}`));
}

In [ ]:
%%writefile notify_test.js
const app = require("./notify_app");
const server = app.listen(0, async () => {
  const base = `http://127.0.0.1:${server.address().port}`;
  const post = (body) => fetch(base + "/notifications", { method: "POST",
    headers: { "Content-Type": "application/json" }, body: JSON.stringify(body) });
  let r = await post({ ticketId: 101, channel: "sms", message: "Your ticket T-101 is now in progress." });
  console.log(r.status, await r.json());
  r = await post({ ticketId: "x", channel: "fax" });
  console.log(r.status, await r.json());
  r = await fetch(base + "/notifications?ticketId=101");
  console.log(r.status, (await r.json()).length, "notification(s) for ticket 101");
  server.close();
});

In [ ]:
# ===== Option A: Google Colab (active) =====
!node notify_test.js

# ===== Option B: VS Code =====
# node notify_test.js
# node notify_app.js   (then test with Postman on http://localhost:4000)

**TODO (SmartDesk step 8):** generate a Postman collection for the notification service with the `req()` helper above (health, valid post, invalid post, list by ticket), and run `npm audit` on the project.

## 9. Check yourself
1. Which file pins exact versions, and which file holds ranges?
2. Why does the router register `/stats` before `/:id`?
3. What did the error middleware return for `?simulate=down`, and why did the server keep running?

---
**Ganbatte (頑張って): *Do your best***

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)